In [1]:
!pip install fugashi unidic-lite pandas scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [71]:
import glob
import os
import re
from collections import Counter
import numpy as np
from fugashi import Tagger
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 形態素解析用の共通インスタンスと品詞定義
tagger = Tagger()
FUNCTION_POS = {"助詞", "助動詞", "接続詞", "連体詞"}
CONTENT_POS = {"名詞", "動詞", "形容詞", "副詞", "感動詞", "形状詞"}


def analyze_text_files(folder_path):
    """指定されたフォルダ内の全テキストファイルを読み込み、形態素解析と各種特徴量（Hapax, Yules_K等）を抽出する"""
    # 数値順にソートしてファイルを取得
    search_path = os.path.join(folder_path, "*.txt")
    filepaths = sorted(
        glob.glob(search_path),
        key=lambda x: int(re.search(r"\d+", os.path.basename(x)).group())
        if re.search(r"\d+", os.path.basename(x))
        else 0,
    )

    files = []
    tokenized_texts = []
    content_texts = []
    function_texts = []
    function_counts = []
    content_counts = []
    function_pers = []
    content_pers = []
    feature_data = []

    for filepath in filepaths:
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
            files.append((filepath, text))

        pos_counter = Counter()
        particle_counter = Counter()
        tokens, content_words, function_words, all_words = [], [], [], []
        word_count = content_count = function_count = 0

        # 記号のカウントと文長計算
        comma_count = text.count("、")
        period_count = text.count("。")
        exclamation_count = text.count("！")
        question_count = text.count("？")

        sentences = re.split(r"[。！？]", text)
        sentence_lengths = [len(s.strip()) for s in sentences if s.strip()]
        avg_sentence_length = (
            np.mean(sentence_lengths) if sentence_lengths else 0
        )

        # 形態素解析
        for word in tagger(text):
            pos = word.feature.pos1
            surface = word.surface

            if pos == "補助記号":
                continue
            if pos == "助詞":
                particle_counter[surface] += 1

            tokens.append(surface)
            all_words.append(surface)
            pos_counter[pos] += 1
            word_count += 1

            if pos in FUNCTION_POS:
                function_words.append(surface)
                function_count += 1
            elif pos in CONTENT_POS:
                content_words.append(surface)
                content_count += 1

        # テキストの結合
        tokenized_texts.append(" ".join(tokens))
        function_texts.append(" ".join(function_words))
        content_texts.append(" ".join(content_words))
        function_counts.append(function_count)
        content_counts.append(content_count)
        
        # 基本割合・指標計算
        if word_count > 0:
            f_per = (function_count / word_count) * 100
            c_per = (content_count / word_count) * 100
            comma_ratio = comma_count / word_count
            period_ratio = period_count / word_count
            ttr = len(set(all_words)) / word_count
            # ===== Hapax / Yule（全・機能・内容）=====
            hapax_all, yule_all = hapax_yule(all_words, word_count)
            hapax_func, yule_func = hapax_yule(function_words, function_count)
            hapax_cont, yule_cont = hapax_yule(content_words, content_count)

        function_pers.append(f_per)
        content_pers.append(c_per)

        total_particle = sum(particle_counter.values())
        particle_ratio = {
            f"助詞_{k}": (v / total_particle if total_particle > 0 else 0)
            for k, v in particle_counter.items()
        }
        pos_ratio = {
            pos: (count / word_count) * 100 for pos, count in pos_counter.items()
        }

        feature_data.append(
            {
                "総単語数": word_count,
                "異なり語数": len(set(all_words)),
                "TTR": ttr,
                "Hapax率_all": hapax_all,
                "Yules_K_all": yule_all,
                "Hapax率_func": hapax_func,
                "Yules_K_func": yule_func,
                "Hapax率_cont": hapax_cont,
                "Yules_K_cont": yule_cont,
                "平均文長": avg_sentence_length,
                "名詞割合": pos_ratio.get("名詞", 0),
                "動詞割合": pos_ratio.get("動詞", 0),
                "形容詞割合": pos_ratio.get("形容詞", 0),
                "助詞割合": pos_ratio.get("助詞", 0),
                "助動詞割合": pos_ratio.get("助動詞", 0),
                "副詞割合": pos_ratio.get("副詞", 0),
                **particle_ratio,
                "読点頻度": comma_ratio,
                "句点頻度": period_ratio,
                "感嘆符数": exclamation_count,
                "疑問符数": question_count,
            }
        )

    return {
        "files": files,
        "tokenized_texts": tokenized_texts,
        "content_texts": content_texts,
        "function_texts": function_texts,
        "function_counts": function_counts,
        "content_counts": content_counts,
        "function_pers": function_pers,
        "content_pers": content_pers,
        "feature_data": feature_data,
    }

def hapax_yule(tokens, word_count):
    if word_count == 0:
        return 0, 0

    freq = Counter(tokens)

    # Hapax率
    hapax_count = sum(1 for v in freq.values() if v == 1)
    hapax_ratio = hapax_count / word_count

    # Yule's K
    freq_counter = Counter(freq.values())
    m1 = word_count
    m2 = sum((f ** 2) * c for f, c in freq_counter.items())

    yules_k = 10000 * ((m2 - m1) / (m1 * m1)) if m1 > 0 else 0

    return hapax_ratio, yules_k

def jpn_word_ngram_analyzer(text):
    """半角スペースを挟ずに日本語の単語N-gram（2-gram, 3-gram）を生成する"""
    tokens = text.split()
    ngrams = []
    for n in [2, 3]:
        for i in range(len(tokens) - n + 1):
            ngrams.append("".join(tokens[i : i + n]))
    return ngrams


def print_analysis_results(
    data,
    X_content,
    content_features,
    Y_function,
    function_features,
    X_function_ngram,
    function_ngram_features,
    X_content_ngram,
    content_ngram_features,
    X_char_ngram,
    char_ngram_features,
    top_k=30,
):
    """解析結果を画面にわかりやすく表示する"""
    for i, (filepath, text) in enumerate(data["files"]):
        function_ngram_values = X_function_ngram.toarray()[i]
        content_ngram_values = X_content_ngram.toarray()[i]
        char_ngram_values = X_char_ngram.toarray()[i]

        # 各種N-gram上位の抽出
        function_ngram_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(
                        function_ngram_features, function_ngram_values
                    )
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )
        content_ngram_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(content_ngram_features, content_ngram_values)
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )
        char_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(char_ngram_features, char_ngram_values)
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )

        data["feature_data"][i].update(
            {
                "function_ngram_top": function_ngram_top,
                "content_ngram_top": content_ngram_top,
                "char_ngram_top": char_top,
            }
        )

        print(f"\n=== ファイル：{filepath} ===")
        for key, value in data["feature_data"][i].items():
            if isinstance(value, float):
                print(f"{key}：{value:.4f}")
            else:
                print(f"{key}：{value}")

        print("\n=== 機能語 ===")
        print(data["function_texts"][i])
        print(f"カウント数：{data['function_counts'][i]}個")
        print(f"割合：{data['function_pers'][i]:.2f}%")
        for word, score in zip(function_features, Y_function.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")

        print("\n=== 内容語 ===")
        print(data["content_texts"][i])
        print(f"カウント数：{data['content_counts'][i]}個")
        print(f"割合：{data['content_pers'][i]:.2f}%")
        for word, score in zip(content_features, X_content.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")


if __name__ == "__main__":
    # フォルダパスを指定してテキスト解析を実行
    analysis_data = analyze_text_files("Human_text")

    if not analysis_data["files"]:
        print("指定されたフォルダにテキストファイルが見つかりませんでした。")
    else:
        # 特徴量抽出（ベクトライズ）の実行
        content_vectorizer = TfidfVectorizer()
        X_content = content_vectorizer.fit_transform(
            analysis_data["content_texts"]
        )
        content_features = content_vectorizer.get_feature_names_out()

        function_vectorizer = TfidfVectorizer()
        Y_function = function_vectorizer.fit_transform(
            analysis_data["function_texts"]
        )
        function_features = function_vectorizer.get_feature_names_out()

        # 機能語の単語N-gram
        function_ngram_vectorizer = CountVectorizer(
            analyzer=jpn_word_ngram_analyzer, max_features=2000
        )
        X_function_ngram = function_ngram_vectorizer.fit_transform(
            analysis_data["function_texts"]
        )
        function_ngram_features = (
            function_ngram_vectorizer.get_feature_names_out()
        )

        # 内容語の単語N-gram
        content_ngram_vectorizer = CountVectorizer(
            analyzer=jpn_word_ngram_analyzer, max_features=2000
        )
        X_content_ngram = content_ngram_vectorizer.fit_transform(
            analysis_data["content_texts"]
        )
        content_ngram_features = (
            content_ngram_vectorizer.get_feature_names_out()
        )

        # 文字N-gram (元の生テキストを使用)
        raw_texts = [text for _, text in analysis_data["files"]]
        char_ngram_vectorizer = CountVectorizer(
            analyzer="char", ngram_range=(3, 5), max_features=3000
        )
        X_char_ngram = char_ngram_vectorizer.fit_transform(raw_texts)
        char_ngram_features = char_ngram_vectorizer.get_feature_names_out()

        # 結果の表示
        print_analysis_results(
            analysis_data,
            X_content,
            content_features,
            Y_function,
            function_features,
            X_function_ngram,
            function_ngram_features,
            X_content_ngram,
            content_ngram_features,
            X_char_ngram,
            char_ngram_features,
        )


=== ファイル：Human_text/Human_text_01.txt ===
総単語数：243
異なり語数：94
TTR：0.3868
Hapax率_all：0.2263
Yules_K_all：199.1566
Hapax率_func：0.0690
Yules_K_func：697.0868
Hapax率_cont：0.3760
Yules_K_cont：151.0400
平均文長：46.0000
名詞割合：31.6872
動詞割合：13.1687
形容詞割合：0.8230
助詞割合：34.1564
助動詞割合：12.3457
副詞割合：0
助詞_は：0.1566
助詞_に：0.0723
助詞_て：0.1205
助詞_と：0.0723
助詞_か：0.0120
助詞_が：0.1566
助詞_で：0.1205
助詞_や：0.0241
助詞_を：0.0964
助詞_の：0.1687
読点頻度：0.0658
句点頻度：0.0412
感嘆符数：0
疑問符数：0
function_ngram_top：{'では': 7, 'とて': 5, 'はの': 5, 'てます': 4, 'のを': 4, 'はのを': 4, 'ではの': 3, 'なを': 3, 'のが': 3, 'ますで': 3, 'をの': 3, 'がで': 2, 'がでは': 2, 'がます': 2, 'がますで': 2, 'たで': 2, 'てた': 2, 'ては': 2, 'とては': 2, 'にが': 2, 'にて': 2, 'にてます': 2, 'のと': 2, 'のとて': 2, 'のな': 2, 'のをの': 2, 'ました': 2, 'ましたで': 2, 'ますが': 2, 'ますでは': 2}
content_ngram_top：{'スマートフォン': 7, 'いスマート': 3, 'いスマートフォン': 3, 'しガラケー': 2, 'する向い': 2, 'する向いい': 2, 'できること': 2, 'コンテンツ生産': 2, '向いい': 2, '向いいスマート': 2, '現代社会': 2, 'いアプリ': 1, 'いアプリ使う': 1, 'い電話': 1, 'い電話カメラ': 1, 'こと出来': 1, 'こと出来パソコン': 1, 'こと挙げ': 1, 'こと挙げ家': 1, 'こ